# Phase 1 — SBERT Text Representation Experiment

This notebook evaluates **SBERT** as the text representation for Software Requirement Prioritization using **LightGBM Ranker** as the fixed baseline ranking model.

**Objective:** Determine whether SBERT produces a strong feature representation for requirement prioritization.

**Model:** `all-MiniLM-L6-v2` → 384-dimensional embeddings (pre-computed in master dataset)

**Ranker:** LightGBM (default parameters, no hyperparameter tuning)


In [1]:
import logging
import os
import json
import joblib
import warnings
import requests

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kendalltau
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import ndcg_score, average_precision_score
import lightgbm as lgb
import mlflow

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
MLFLOW_URI = "https://mlflow.smbgarasibmw.my.id"

server_online = False
try:
    resp = requests.get(MLFLOW_URI, timeout=3)
    if resp.status_code == 200:
        mlflow.set_tracking_uri(MLFLOW_URI)
        mlflow.set_experiment("Phase1_SBERT_Text_Representation")
        logger_message = f"Successfully connected to MLflow tracking server: {MLFLOW_URI}"
        server_online = True
    else:
        logger_message = f"MLflow server returned HTTP {resp.status_code}. Using local MLflow store (./mlruns)."
except Exception as exc:
    logger_message = f"MLflow server unreachable ({exc}). Using local MLflow store (./mlruns)."

if not server_online:
    mlflow.set_tracking_uri("file:./mlruns")
    mlflow.set_experiment("Phase1_SBERT_Text_Representation")

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)
logger.info(logger_message)

2026/09/22 07:25:19 INFO mlflow.tracking.fluent: Experiment with name 'Phase1_SBERT_Text_Representation' does not exist. Creating a new experiment.
2026-09-22 07:25:21,510 - INFO - Successfully connected to MLflow tracking server: https://mlflow.smbgarasibmw.my.id


In [2]:
# Configuration
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
EMBEDDING_DIM = 384  # Expected SBERT dimension
RANDOM_STATE = 42
TEST_SIZE = 0.3
OUTPUT_DIR = "outputs/phase1_sbert"
DATASET_PATH = "dataset/sentence_embedding_training.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)
logger.info(f"Output directory: {OUTPUT_DIR}")

2026-09-22 07:25:21,520 - INFO - Output directory: outputs/phase1_sbert


## STEP 1 — Load Dataset

Load the master dataset from `dataset/sentence_embedding.csv`. This is the source of truth containing all original metadata and pre-computed SBERT embeddings.


In [3]:
df = pd.read_csv(DATASET_PATH)
logger.info(f"Dataset shape: {df.shape}")
logger.info(f"Columns: {list(df.columns)}")

# Detect requirement text column
text_keywords = ["requirement", "text", "description", "sentence", "story", "content"]
text_cols = [c for c in df.columns if any(k in c.lower() for k in text_keywords)]
if text_cols:
    logger.info(f"Detected requirement text column(s): {text_cols}")
else:
    logger.info("No explicit requirement text column found. Using pre-computed embeddings.")

# Validate required columns
required_cols = {"id", "project_id", "value", "effort", "risk", "stakeholder_priority", "rank", "type_FR", "type_NFR"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

embed_cols = [c for c in df.columns if c.startswith("embedding_")]
if not embed_cols:
    raise ValueError("No embedding columns found in dataset. "
                     "Expected pre-computed SBERT embeddings (embedding_0..embedding_N)")

logger.info(f"Found {len(embed_cols)} embedding columns ({embed_cols[0]} to {embed_cols[-1]})")
logger.info("All required columns present.")

display(df.head(3))
display(df.describe())

2026-09-22 07:25:21,584 - INFO - Dataset shape: (951, 394)
2026-09-22 07:25:21,586 - INFO - Columns: ['id', 'project_id', 'value', 'effort', 'risk', 'stakeholder_priority', 'priority_score', 'rank', 'embedding_0', 'embedding_1', 'embedding_2', 'embedding_3', 'embedding_4', 'embedding_5', 'embedding_6', 'embedding_7', 'embedding_8', 'embedding_9', 'embedding_10', 'embedding_11', 'embedding_12', 'embedding_13', 'embedding_14', 'embedding_15', 'embedding_16', 'embedding_17', 'embedding_18', 'embedding_19', 'embedding_20', 'embedding_21', 'embedding_22', 'embedding_23', 'embedding_24', 'embedding_25', 'embedding_26', 'embedding_27', 'embedding_28', 'embedding_29', 'embedding_30', 'embedding_31', 'embedding_32', 'embedding_33', 'embedding_34', 'embedding_35', 'embedding_36', 'embedding_37', 'embedding_38', 'embedding_39', 'embedding_40', 'embedding_41', 'embedding_42', 'embedding_43', 'embedding_44', 'embedding_45', 'embedding_46', 'embedding_47', 'embedding_48', 'embedding_49', 'embedding_

,id,project_id,value,effort,risk,stakeholder_priority,priority_score,rank,embedding_0,embedding_1,...,embedding_376,embedding_377,embedding_378,embedding_379,embedding_380,embedding_381,embedding_382,embedding_383,type_FR,type_NFR
0,REQ-01,P1,3,2,1,3,1.3,18,0.030996,0.082017,...,0.018806,-0.011850,-0.083696,-0.032730,0.130055,0.007294,0.012843,-0.061484,1,0
1,REQ-02,P1,3,2,1,3,1.3,18,-0.008261,0.051770,...,-0.026357,-0.051586,-0.117152,-0.122079,0.058768,0.004795,0.039507,-0.062729,1,0
2,REQ-03,P1,3,2,1,3,1.3,18,-0.032415,0.011653,...,-0.024535,-0.056196,-0.137691,-0.146946,0.075015,-0.013269,0.052900,-0.096478,1,0


,value,effort,risk,stakeholder_priority,priority_score,rank,embedding_0,embedding_1,embedding_2,embedding_3,...,embedding_376,embedding_377,embedding_378,embedding_379,embedding_380,embedding_381,embedding_382,embedding_383,type_FR,type_NFR
count,951.000000,951.000000,951.000000,951.000000,951.000000,951.000000,951.000000,951.000000,951.000000,951.000000,...,951.000000,951.000000,951.000000,951.000000,951.000000,951.000000,951.000000,951.000000,951.000000,951.000000
mean,3.929548,2.935857,2.930599,3.927445,1.473712,46.527865,-0.019313,0.030853,-0.015226,-0.009741,...,0.025499,-0.015379,0.012476,0.004147,0.026073,0.010335,0.004467,-0.007670,0.809674,0.190326
std,0.857340,0.914841,1.262170,0.860840,0.391573,55.045273,0.046978,0.049486,0.045020,0.044790,...,0.039501,0.043061,0.053277,0.047021,0.056529,0.046075,0.051919,0.043413,0.392765,0.392765
min,2.000000,1.000000,1.000000,2.000000,0.200000,1.000000,-0.157287,-0.139118,-0.159155,-0.152741,...,-0.095749,-0.155481,-0.147005,-0.146946,-0.134916,-0.130625,-0.131922,-0.150628,0.000000,0.000000
25%,3.000000,2.000000,2.000000,3.000000,1.200000,7.000000,-0.052250,-0.003293,-0.044160,-0.039791,...,-0.002190,-0.045175,-0.021456,-0.027205,-0.012189,-0.021531,-0.032111,-0.036028,1.000000,0.000000
50%,4.000000,3.000000,3.000000,4.000000,1.500000,19.000000,-0.021965,0.028603,-0.015182,-0.008184,...,0.026176,-0.015269,0.012947,0.003842,0.026110,0.009224,0.005948,-0.006815,1.000000,0.000000
75%,5.000000,3.000000,4.000000,5.000000,1.700000,74.000000,0.014505,0.066606,0.014621,0.020323,...,0.053998,0.015086,0.046923,0.037422,0.065233,0.040121,0.041843,0.020852,1.000000,0.000000
max,5.000000,5.000000,5.000000,5.000000,2.700000,232.000000,0.130875,0.214671,0.137820,0.132734,...,0.150365,0.132034,0.185025,0.159019,0.214406,0.161515,0.141851,0.129366,1.000000,1.000000


## STEP 2 — Data Validation

Perform validation checks: missing values, duplicates, invalid project IDs, invalid rank values, and data types.


In [4]:
validation_report = {}

# Missing values
missing_counts = df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]
validation_report["missing_values"] = len(missing_cols)
if len(missing_cols) > 0:
    logger.warning(f"Columns with missing values:\n{missing_cols}")
else:
    logger.info("No missing values found.")

# Duplicate rows
dup_rows = df.duplicated().sum()
validation_report["duplicate_rows"] = dup_rows
if dup_rows > 0:
    logger.warning(f"Duplicate rows: {dup_rows}")
else:
    logger.info("No duplicate rows found.")

# Duplicate requirements (by id)
dup_ids = df["id"].duplicated().sum()
validation_report["duplicate_ids"] = dup_ids
if dup_ids > 0:
    logger.warning(f"Duplicate requirement IDs: {dup_ids}")

# Invalid project_id
invalid_pid = df["project_id"].isnull().sum() | (df["project_id"].astype(str).str.strip() == "").sum()
validation_report["invalid_project_ids"] = int(invalid_pid)
if invalid_pid > 0:
    logger.warning(f"Invalid project IDs: {invalid_pid}")

# Invalid rank (should be numeric, non-negative)
invalid_rank = (~pd.to_numeric(df["rank"], errors="coerce").notna()).sum()
validation_report["invalid_rank"] = int(invalid_rank)
if invalid_rank > 0:
    logger.warning(f"Invalid rank values: {invalid_rank}")

# Data types
validation_report["dtypes"] = {c: str(dt) for c, dt in df.dtypes.items()}

logger.info("=== Validation Report ===")
for k, v in validation_report.items():
    logger.info(f"  {k}: {v}")


2026-09-22 07:25:21,902 - INFO - No missing values found.
2026-09-22 07:25:21,929 - INFO - No duplicate rows found.
2026-09-22 07:25:21,936 - INFO - === Validation Report ===
2026-09-22 07:25:21,936 - INFO -   missing_values: 0
2026-09-22 07:25:21,938 - INFO -   duplicate_rows: 0
2026-09-22 07:25:21,938 - INFO -   duplicate_ids: 0
2026-09-22 07:25:21,938 - INFO -   invalid_project_ids: 0
2026-09-22 07:25:21,938 - INFO -   invalid_rank: 0
2026-09-22 07:25:21,940 - INFO -   dtypes: {'id': 'object', 'project_id': 'object', 'value': 'int64', 'effort': 'int64', 'risk': 'int64', 'stakeholder_priority': 'int64', 'priority_score': 'float64', 'rank': 'int64', 'embedding_0': 'float64', 'embedding_1': 'float64', 'embedding_2': 'float64', 'embedding_3': 'float64', 'embedding_4': 'float64', 'embedding_5': 'float64', 'embedding_6': 'float64', 'embedding_7': 'float64', 'embedding_8': 'float64', 'embedding_9': 'float64', 'embedding_10': 'float64', 'embedding_11': 'float64', 'embedding_12': 'float64', 

## STEP 3 — SBERT Text Representation

The master dataset contains pre-computed SBERT embeddings generated using `all-MiniLM-L6-v2`. These are 384-dimensional sentence embeddings stored in columns `embedding_0` through `embedding_383`.

**Note:** The actual embedding dimension in the dataset is determined dynamically.


In [5]:
embed_cols = sorted([c for c in df.columns if c.startswith("embedding_")],
                      key=lambda x: int(x.split("_")[1]))

actual_dim = len(embed_cols)
logger.info(f"Embedding dimension: {actual_dim}")

embedding_matrix = df[embed_cols].values
logger.info(f"Embedding matrix shape: {embedding_matrix.shape}")

logger.info(f"First embedding vector (first 10 dims): {embedding_matrix[0, :10]}")
logger.info(f"First embedding vector shape: {embedding_matrix[0].shape}")

# Quick distribution check
logger.info(f"Embedding value range: [{embedding_matrix.min():.4f}, {embedding_matrix.max():.4f}]")
logger.info(f"Mean: {embedding_matrix.mean():.6f}, Std: {embedding_matrix.std():.6f}")


2026-09-22 07:25:21,948 - INFO - Embedding dimension: 384
2026-09-22 07:25:21,953 - INFO - Embedding matrix shape: (951, 384)
2026-09-22 07:25:21,953 - INFO - First embedding vector (first 10 dims): [ 0.03099552  0.08201721 -0.03240236  0.04377936  0.02469208 -0.00162476
 -0.02168863 -0.03060964  0.06035456  0.02331645]
2026-09-22 07:25:21,953 - INFO - First embedding vector shape: (384,)
2026-09-22 07:25:21,955 - INFO - Embedding value range: [-0.2546, 0.2402]
2026-09-22 07:25:21,958 - INFO - Mean: 0.000666, Std: 0.051027


## STEP 4 — Feature Fusion

Create the final feature matrix by combining:
- **Numerical features:** `value`, `effort`, `risk`, `stakeholder_priority`
- **Text features:** `embedding_0` ... `embedding_N`

**Excluded:**
- `id`, `priority_score` (target leakage), `rank` (target), `project_id` (query group)


In [6]:
num_features = ["value", "effort", "risk", "stakeholder_priority"]
feature_cols = num_features + embed_cols

logger.info(f"Numerical features: {num_features}")
logger.info(f"Total feature dimension: {len(feature_cols)} (4 numerical + {len(embed_cols)} embedding)")

X = df[feature_cols].copy()
logger.info(f"Feature matrix shape: {X.shape}")
display(X.head(3))


2026-09-22 07:25:21,967 - INFO - Numerical features: ['value', 'effort', 'risk', 'stakeholder_priority']
2026-09-22 07:25:21,967 - INFO - Total feature dimension: 388 (4 numerical + 384 embedding)
2026-09-22 07:25:21,972 - INFO - Feature matrix shape: (951, 388)


,value,effort,risk,stakeholder_priority,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,...,embedding_374,embedding_375,embedding_376,embedding_377,embedding_378,embedding_379,embedding_380,embedding_381,embedding_382,embedding_383
0,3,2,1,3,0.030996,0.082017,-0.032402,0.043779,0.024692,-0.001625,...,-0.057721,0.000974,0.018806,-0.011850,-0.083696,-0.032730,0.130055,0.007294,0.012843,-0.061484
1,3,2,1,3,-0.008261,0.051770,-0.009457,0.019407,-0.009472,0.069842,...,-0.047565,-0.024281,-0.026357,-0.051586,-0.117152,-0.122079,0.058768,0.004795,0.039507,-0.062729
2,3,2,1,3,-0.032415,0.011653,-0.014935,0.021126,0.021682,0.049995,...,-0.062729,-0.010167,-0.024535,-0.056196,-0.137691,-0.146946,0.075015,-0.013269,0.052900,-0.096478


## STEP 5 — Feature Encoding

One-hot encode the `type` column (FR/NFR) and append to the feature matrix.


In [7]:
encoded_type = df[["type_FR", "type_NFR"]].apply(pd.to_numeric, errors="coerce").fillna(0).astype(float)
logger.info(f"Using type indicator columns: {list(encoded_type.columns)}")

X = pd.concat([X, encoded_type], axis=1)
logger.info(f"Feature matrix shape after encoding: {X.shape}")

display(X.head(3))

2026-09-22 07:25:21,996 - INFO - Using type indicator columns: ['type_FR', 'type_NFR']
2026-09-22 07:25:22,000 - INFO - Feature matrix shape after encoding: (951, 390)


,value,effort,risk,stakeholder_priority,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,...,embedding_376,embedding_377,embedding_378,embedding_379,embedding_380,embedding_381,embedding_382,embedding_383,type_FR,type_NFR
0,3,2,1,3,0.030996,0.082017,-0.032402,0.043779,0.024692,-0.001625,...,0.018806,-0.011850,-0.083696,-0.032730,0.130055,0.007294,0.012843,-0.061484,1.0,0.0
1,3,2,1,3,-0.008261,0.051770,-0.009457,0.019407,-0.009472,0.069842,...,-0.026357,-0.051586,-0.117152,-0.122079,0.058768,0.004795,0.039507,-0.062729,1.0,0.0
2,3,2,1,3,-0.032415,0.011653,-0.014935,0.021126,0.021682,0.049995,...,-0.024535,-0.056196,-0.137691,-0.146946,0.075015,-0.013269,0.052900,-0.096478,1.0,0.0


## STEP 6 — Query Group Construction

Prepare the dataset for Learning to Rank:
- **Query/group identifier:** `project_id`
- **Ranking label:** `rank`
- Verify every project contains multiple requirements.


In [8]:
# Transform rank into lambdarank-compatible labels
# Within each group: dense rank (0 = worst, N-1 = best)
# Lower original rank = higher priority → higher label
df["label"] = df.groupby("project_id")["rank"].transform(
    lambda x: x.rank(method="dense", ascending=False).astype(int) - 1
)

y = df["label"].values
groups = df["project_id"].values
query_ids = df["project_id"]

group_sizes = query_ids.value_counts()
logger.info(f"Number of query groups (projects): {len(group_sizes)}")
logger.info(f"Group size stats:\n{group_sizes.describe()}")

single_req_groups = (group_sizes == 1).sum()
if single_req_groups > 0:
    logger.warning(f"Groups with only 1 requirement: {single_req_groups} — these cannot be ranked")

# Compute group counts for LightGBM
group_counts = group_sizes.sort_index().values  # sorted by project_id
group_order = group_sizes.index.sort_values()
logger.info(f"Group sizes: {group_counts[:10]}... (showing first 10)")

# Validate label distribution
logger.info(f"Label value range: [{y.min()}, {y.max()}]")
logger.info(f"Unique labels: {sorted(np.unique(y))[:20]}...")


2026-09-22 07:25:22,031 - INFO - Number of query groups (projects): 30
2026-09-22 07:25:22,035 - INFO - Group size stats:
count     30.000000
mean      31.700000
std       47.651863
min        3.000000
25%       10.000000
50%       17.500000
75%       23.500000
max      231.000000
Name: count, dtype: float64
2026-09-22 07:25:22,035 - INFO - Group sizes: [47  8 18 10 19  9  3 20 25 20]... (showing first 10)
2026-09-22 07:25:22,035 - INFO - Label value range: [0, 20]
2026-09-22 07:25:22,037 - INFO - Unique labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]...


## STEP 7 — Train/Test Split

Use `GroupShuffleSplit` to ensure the same project never appears in both train and test sets. Fixed `random_state=42` for reproducibility.


In [9]:
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y[train_idx]
y_test = y[test_idx]
groups_train = groups[train_idx]
groups_test = groups[test_idx]

# Recompute group counts for train/test
train_group_counts = pd.Series(groups_train).value_counts().sort_index().values
test_group_counts = pd.Series(groups_test).value_counts().sort_index().values

logger.info(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
logger.info(f"Train groups: {len(np.unique(groups_train))}, Test groups: {len(np.unique(groups_test))}")

# Verify no overlap
train_projects = set(np.unique(groups_train))
test_projects = set(np.unique(groups_test))
overlap = train_projects & test_projects
if overlap:
    raise ValueError(f"Project overlap between train and test: {overlap}")
logger.info("No project overlap between train and test sets. Split is clean.")


2026-09-22 07:25:22,073 - INFO - Train size: 649, Test size: 302
2026-09-22 07:25:22,074 - INFO - Train groups: 21, Test groups: 9
2026-09-22 07:25:22,076 - INFO - No project overlap between train and test sets. Split is clean.


## STEP 8 — LightGBM Ranker Training

Train a baseline LightGBM Ranker with reasonable default parameters. No hyperparameter tuning — the goal is to evaluate SBERT representation quality, not to optimize the ranker.


In [10]:
max_group_size = df.groupby("project_id").size().max()
num_leaves = min(max_group_size, 255)

ranker = lgb.LGBMRanker(
    objective="lambdarank",
    boosting_type="gbdt",
    n_estimators=100,
    num_leaves=num_leaves,
    learning_rate=0.1,
    min_child_samples=10,
    random_state=RANDOM_STATE,
    verbose=-1
)

logger.info("Training LightGBM Ranker...")
ranker.fit(
    X_train, y_train,
    group=train_group_counts,
    eval_set=[(X_test, y_test)],
    eval_group=[test_group_counts],
    eval_metric=["ndcg"],
    callbacks=[lgb.log_evaluation(0)]
)
logger.info("Training complete.")

logger.info(f"Feature importances (top 10):\n"
            f"{pd.Series(ranker.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)}")


2026-09-22 07:25:22,087 - INFO - Training LightGBM Ranker...
2026-09-22 07:25:25,304 - INFO - Training complete.
2026-09-22 07:25:25,304 - INFO - Feature importances (top 10):
effort           121
value            108
risk              87
embedding_252     58
embedding_102     48
embedding_38      48
embedding_205     43
embedding_51      35
embedding_94      35
embedding_366     33
dtype: int32


## STEP 9 — Model Evaluation

Evaluate using:
- **NDCG@5, NDCG@10** — Normalized Discounted Cumulative Gain
- **MAP** — Mean Average Precision
- **Spearman Rank Correlation**
- **Kendall Tau**


In [11]:
y_pred = ranker.predict(X_test)

# Compute NDCG per query group, then average
test_project_ids = groups_test
ndcg5_scores = []
ndcg10_scores = []
map_per_group = []

for pid in np.unique(test_project_ids):
    mask = test_project_ids == pid
    y_true_group = y_test[mask]
    y_pred_group = y_pred[mask]
    n = len(y_true_group)

    # NDCG
    if n >= 5:
        k5 = min(5, n)
        ndcg5_scores.append(ndcg_score(y_true_group.reshape(1, -1),
                                        y_pred_group.reshape(1, -1), k=k5))
    if n >= 10:
        k10 = min(10, n)
        ndcg10_scores.append(ndcg_score(y_true_group.reshape(1, -1),
                                         y_pred_group.reshape(1, -1), k=k10))

    # MAP (binarize: top half of labels = relevant)
    if len(np.unique(y_true_group)) > 1:
        threshold = y_true_group.max() * 0.5
        y_bin = (y_true_group >= threshold).astype(int)
        if y_bin.sum() > 0 and y_bin.sum() < len(y_bin):
            map_per_group.append(average_precision_score(y_bin, y_pred_group))

ndcg5 = float(np.mean(ndcg5_scores)) if ndcg5_scores else 0.0
ndcg10 = float(np.mean(ndcg10_scores)) if ndcg10_scores else 0.0
map_score = float(np.mean(map_per_group)) if map_per_group else 0.0

spearman_corr, spearman_p = spearmanr(y_test, y_pred)
kendall_corr, kendall_p = kendalltau(y_test, y_pred)

metrics = {
    "NDCG_at_5": round(float(ndcg5), 6),
    "NDCG_at_10": round(float(ndcg10), 6),
    "MAP": round(float(map_score), 6),
    "Spearman": round(float(spearman_corr), 6),
    "Spearman_pvalue": float(spearman_p),
    "KendallTau": round(float(kendall_corr), 6),
    "KendallTau_pvalue": float(kendall_p)
}

logger.info("=== Evaluation Metrics ===")
for k, v in metrics.items():
    logger.info(f"  {k}: {v}")

display(pd.DataFrame([metrics]))


2026-09-22 07:25:25,357 - INFO - === Evaluation Metrics ===
2026-09-22 07:25:25,358 - INFO -   NDCG_at_5: 0.8858
2026-09-22 07:25:25,359 - INFO -   NDCG_at_10: 0.915018
2026-09-22 07:25:25,360 - INFO -   MAP: 0.918191
2026-09-22 07:25:25,361 - INFO -   Spearman: 0.37721
2026-09-22 07:25:25,362 - INFO -   Spearman_pvalue: 1.2007077164523649e-11
2026-09-22 07:25:25,362 - INFO -   KendallTau: 0.268645
2026-09-22 07:25:25,364 - INFO -   KendallTau_pvalue: 1.9577250608488018e-11


,NDCG_at_5,NDCG_at_10,MAP,Spearman,Spearman_pvalue,KendallTau,KendallTau_pvalue
0,0.8858,0.915018,0.918191,0.37721,1.200708e-11,0.268645,1.957725e-11


## STEP 10 — Save Outputs

Save all experiment outputs to `outputs/phase1_sbert/`:
- Trained LightGBM model
- Generated SBERT dataset (for Phase 2 input)
- Evaluation metrics (JSON)
- Predictions
- Feature list


In [12]:
# 1. Save trained model
model_path = os.path.join(OUTPUT_DIR, "lgbm_ranker.pkl")
joblib.dump(ranker, model_path)
logger.info(f"Model saved: {model_path}")

# 2. Save generated dataset (Phase 2 input)
dataset_out = X.copy()
dataset_out["label"] = y
dataset_out["rank"] = df["rank"].values
dataset_out["project_id"] = groups
dataset_path = os.path.join(OUTPUT_DIR, "sbert_dataset.csv")
dataset_out.to_csv(dataset_path, index=False)
logger.info(f"Generated dataset saved: {dataset_path} (shape: {dataset_out.shape})")

# 3. Save evaluation metrics
metrics_path = os.path.join(OUTPUT_DIR, "evaluation_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
logger.info(f"Metrics saved: {metrics_path}")

# 4. Save predictions
predictions_df = pd.DataFrame({
    "project_id": groups_test,
    "true_label": y_test,
    "predicted_score": y_pred
})
pred_path = os.path.join(OUTPUT_DIR, "predictions.csv")
predictions_df.to_csv(pred_path, index=False)
logger.info(f"Predictions saved: {pred_path}")

# 5. Save feature list
features_path = os.path.join(OUTPUT_DIR, "feature_list.txt")
with open(features_path, "w") as f:
    f.write("\n".join(X.columns.tolist()))
logger.info(f"Feature list saved: {features_path}")


2026-09-22 07:25:25,420 - INFO - Model saved: outputs/phase1_sbert\lgbm_ranker.pkl


2026-09-22 07:25:25,689 - INFO - Generated dataset saved: outputs/phase1_sbert\sbert_dataset.csv (shape: (951, 393))
2026-09-22 07:25:25,689 - INFO - Metrics saved: outputs/phase1_sbert\evaluation_metrics.json
2026-09-22 07:25:25,689 - INFO - Predictions saved: outputs/phase1_sbert\predictions.csv
2026-09-22 07:25:25,689 - INFO - Feature list saved: outputs/phase1_sbert\feature_list.txt


## STEP 11 — MLflow Logging

Track the experiment using MLflow: log parameters, metrics, and artifacts.


In [13]:
with mlflow.start_run(run_name="SBERT_all-MiniLM-L6-v2_LightGBM") as run:
    # Log parameters
    mlflow.log_param("embedding_model", EMBEDDING_MODEL_NAME)
    mlflow.log_param("embedding_dimension", actual_dim)
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    mlflow.log_param("num_train_groups", len(np.unique(groups_train)))
    mlflow.log_param("num_test_groups", len(np.unique(groups_test)))
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_param("test_size_ratio", TEST_SIZE)
    mlflow.log_param("num_features", X.shape[1])
    mlflow.log_param("ranker_objective", "lambdarank")
    mlflow.log_param("ranker_n_estimators", 100)
    mlflow.log_param("ranker_num_leaves", num_leaves)

    # Log metrics
    mlflow.log_metrics(metrics)

    # Log artifacts
    mlflow.log_artifact(model_path, artifact_path="model")
    mlflow.log_artifact(metrics_path, artifact_path="metrics")
    mlflow.log_artifact(features_path, artifact_path="features")
    mlflow.log_artifact(dataset_path, artifact_path="dataset")
    mlflow.log_artifact(pred_path, artifact_path="predictions")

    logger.info(f"MLflow run ID: {run.info.run_id}")
    logger.info(f"MLflow experiment logged successfully.")


2026-09-22 07:25:30,193 - INFO - MLflow run ID: f4e72e0575a34803b6e2ec8b884a4bf7
2026-09-22 07:25:30,195 - INFO - MLflow experiment logged successfully.


🏃 View run SBERT_all-MiniLM-L6-v2_LightGBM at: https://mlflow.smbgarasibmw.my.id/#/experiments/3/runs/f4e72e0575a34803b6e2ec8b884a4bf7
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/3


## Experiment Summary

### What was done
1. **Dataset:** Loaded master dataset with pre-computed SBERT embeddings
2. **Features:** 4 numerical features + embedding dimensions + one-hot encoded type
3. **Model:** LightGBM Ranker (Lambdarank objective, default parameters)
4. **Split:** GroupShuffleSplit (30% test, no project overlap)
5. **Evaluation:** NDCG@5, NDCG@10, MAP, Spearman, Kendall Tau


In [14]:
summary_data = {
    "Metric": ["NDCG@5", "NDCG@10", "MAP", "Spearman rho", "Spearman p-value", "Kendall tau", "Kendall tau p-value"],
    "Value": [
        metrics["NDCG_at_5"],
        metrics["NDCG_at_10"],
        metrics["MAP"],
        metrics["Spearman"],
        metrics["Spearman_pvalue"],
        metrics["KendallTau"],
        metrics["KendallTau_pvalue"]
    ]
}
summary_df = pd.DataFrame(summary_data)

logger.info("=== Experiment Summary ===")
logger.info(f"Embedding model: {EMBEDDING_MODEL_NAME}")
logger.info(f"Embedding dimension: {actual_dim}")
logger.info(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
logger.info(f"Train groups: {len(np.unique(groups_train))}, Test groups: {len(np.unique(groups_test))}")
logger.info(f"Total features: {X.shape[1]}")
logger.info(f"\n{summary_df.to_string(index=False)}")

display(summary_df)


2026-09-22 07:25:30,632 - INFO - === Experiment Summary ===
2026-09-22 07:25:30,633 - INFO - Embedding model: all-MiniLM-L6-v2
2026-09-22 07:25:30,633 - INFO - Embedding dimension: 384
2026-09-22 07:25:30,634 - INFO - Train size: 649, Test size: 302
2026-09-22 07:25:30,634 - INFO - Train groups: 21, Test groups: 9
2026-09-22 07:25:30,634 - INFO - Total features: 390
2026-09-22 07:25:30,634 - INFO - 
             Metric        Value
             NDCG@5 8.858000e-01
            NDCG@10 9.150180e-01
                MAP 9.181910e-01
       Spearman rho 3.772100e-01
   Spearman p-value 1.200708e-11
        Kendall tau 2.686450e-01
Kendall tau p-value 1.957725e-11


,Metric,Value
0,NDCG@5,8.858000e-01
1,NDCG@10,9.150180e-01
2,MAP,9.181910e-01
3,Spearman rho,3.772100e-01
4,Spearman p-value,1.200708e-11
5,Kendall tau,2.686450e-01
6,Kendall tau p-value,1.957725e-11


### Outputs
All artifacts saved to `outputs/phase1_sbert/`. The generated dataset is ready for Phase 2 (Ranking Algorithm Experiment).

---
*Phase 1 — SBERT Text Representation Experiment complete.*